# 动态工具注册表

dict[str, Callable[[dict], str]]

一个字典：
key 是工具名 str；
value 是一个（回调）函数；
这个（回调）函数接收 dict（字典类型的参数），返回 str。

最小可运行示例
下面是一个最小、可运行、可验证的例子：

In [1]:
from typing import Callable

# 动态工具实现表：name -> callable(args dict) -> str
DYNAMIC_TOOL_IMPL: dict[str, Callable[[dict], str]] = {}


def hello(args: dict) -> str:
    name = args["name"]
    return f"Hello, {name}"


def add(args: dict) -> str:
    a = args["a"]
    b = args["b"]
    return str(a + b)


DYNAMIC_TOOL_IMPL["hello"] = hello
DYNAMIC_TOOL_IMPL["add"] = add


result1 = DYNAMIC_TOOL_IMPL["hello"]({"name": "Alice"})
result2 = DYNAMIC_TOOL_IMPL["add"]({"a": 1, "b": 2})

print(result1)
print(result2)


Hello, Alice
3


带参数装饰器的形式：
register_tool_impl(name) -> deco（内部函数） -> deco(fn)（调用时） -> fn（函数本身）
构成这种格式就是装饰器，就可以用来修饰函数

完整过程
```Python
@register_tool_impl("add")
def add(a, b):
    return a + b
```
大致等价于：

```Python
def add(a, b):    return a + b
add = register_tool_impl("add")(add)
```
也就是：
```Python
deco = register_tool_impl("add")
add = deco(add)
```

In [1]:
DYNAMIC_TOOL_IMPL = {}


def register_tool_impl(name: str):
    def deco(fn):
        DYNAMIC_TOOL_IMPL[name] = fn
        return fn
    return deco


@register_tool_impl("add")
def add(a, b):
    return a + b


@register_tool_impl("hello")
def hello(name):
    return f"Hello, {name}!"


print(DYNAMIC_TOOL_IMPL)

print(DYNAMIC_TOOL_IMPL["add"](1, 2))
print(DYNAMIC_TOOL_IMPL["hello"]("Alice"))

print(add(10, 20))
print(hello("Bob"))


{'add': <function add at 0x000002F025E17100>, 'hello': <function hello at 0x000002F025E171A0>}
3
Hello, Alice!
30
Hello, Bob!
